# AutoML Tutorial: H2O AutoML
> **Run this notebook in Google Colab or Jupyter to learn about automated machine learning (AutoML) using H2O AutoML.**

## 1. Introduction
Automated Machine Learning (AutoML) simplifies the end-to-end ML workflow by:
- **Feature preprocessing**
- **Model selection**
- **Hyperparameter optimization**
- **Ensembling**

In this tutorial, we'll focus solely on **H2O AutoML**, a scalable AutoML framework supporting both regression and classification.


At the end, you'll complete an exercise applying AutoML.



## 2. Setup & Installation

In [2]:
!pip install --quiet jedi
!pip install --quiet h2o
!pip install --quiet 'thinc<8.3.6'

## Regression on California Housing


In [3]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


## 3. Regression Example: California Housing

In [4]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import h2o
from h2o.automl import H2OAutoML
from sklearn.metrics import mean_squared_error, r2_score

In [5]:
# Initialize H2O
h2o.init(max_mem_size="2G", nthreads=-1)

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "17.0.19" 2026-04-21; OpenJDK Runtime Environment (build 17.0.19+10-1-22.04.2-Ubuntu); OpenJDK 64-Bit Server VM (build 17.0.19+10-1-22.04.2-Ubuntu, mixed mode, sharing)
  Starting server from /usr/local/lib/python3.12/dist-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmpdc3kyt9j
  JVM stdout: /tmp/tmpdc3kyt9j/h2o_unknownUser_started_from_python.out
  JVM stderr: /tmp/tmpdc3kyt9j/h2o_unknownUser_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,06 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.11
H2O_cluster_version_age:,2 months and 6 days
H2O_cluster_name:,H2O_from_python_unknownUser_2avmm0
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,1.981 Gb
H2O_cluster_total_cores:,2
H2O_cluster_allowed_cores:,2
H2O_cluster_status:,"locked, healthy"


In [6]:
# Load data
data = fetch_california_housing(as_frame=True)
X = data.data
y = data.target.rename('target')

In [7]:
# Create H2OFrame
df = h2o.H2OFrame(pd.concat([X, y], axis=1))

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [8]:
# Split into train/test
train, test = df.split_frame(ratios=[0.8], seed=42)

### 3.1 Run H2O AutoML for Regression

In [9]:
aml_reg = H2OAutoML(
    max_runtime_secs=300,
    max_models=20,
    seed=42,
    nfolds=5,
    project_name="california_regression"
)
aml_reg.train(x=X.columns.tolist(), y='target', training_frame=train)

AutoML progress: |███████████████████████████████████████████████████████████████| (done) 100%


Model Details
=============
H2OGradientBoostingEstimator : Gradient Boosting Machine
Model Key: GBM_4_AutoML_1_20260729_02346


Model Summary: 
    number_of_trees    number_of_internal_trees    model_size_in_bytes    min_depth    max_depth    mean_depth    min_leaves    max_leaves    mean_leaves
--  -----------------  --------------------------  ---------------------  -----------  -----------  ------------  ------------  ------------  -------------
    99                 99                          277095                 10           10           10            41            451           218.404

ModelMetricsRegression: gbm
** Reported on train data. **

MSE: 0.07397325656727911
RMSE: 0.2719802503257895
MAE: 0.18538668936475916
RMSLE: 0.0835524573825608
Mean Residual Deviance: 0.07397325656727911

ModelMetricsRegression: gbm
** Reported on cross-validation data. **

MSE: 0.20714453019161605
RMSE: 0.4551313329047079
MAE: 0.29676202261609314
RMSLE: 0.13681969786908746
Mean Residual Deviance: 0.20714453019161605

Cross-Validation Metrics Summary: 
                        mean      sd           cv_1_valid    cv_2_valid    cv_3_valid    cv_4_valid    cv_5_valid
----------------------  --------  -----------  ------------  ------------  ------------  ------------  ------------
aic                     nan       0            nan           nan           nan           nan           nan
loglikelihood           nan       0            nan           nan           nan           nan           nan
mae                     0.296732  0.00322323   0.299136      0.292047      0.294765      0.298175      0.299539
mean_residual_deviance  0.207227  0.00613069   0.217706      0.204902      0.203079      0.207449      0.203
mse                     0.207227  0.00613069   0.217706      0.204902      0.203079      0.207449      0.203
r2                      0.844695  0.00317222   0.840185      0.845816      0.847472      0.842682      0.847321
residual_deviance       0.207227  0.00613069   0.217706      0.204902      0.203079      0.207449      0.203
rmse                    0.455183  0.00668183   0.46659       0.452661      0.450643      0.455466      0.450555
rmsle                   0.136859  0.000486415  0.137507      0.136554      0.136472      0.136504      0.137256

Scoring History: 
     timestamp            duration    number_of_trees    training_rmse        training_mae         training_deviance
---  -------------------  ----------  -----------------  -------------------  -------------------  -------------------
     2026-07-29 00:26:40  14.674 sec  0.0                1.1550860447424607   0.9126682923030268   1.334223770758782
     2026-07-29 00:26:40  14.804 sec  5.0                0.8099859863741191   0.6333660448978525   0.6560772981224546
     2026-07-29 00:26:40  14.938 sec  10.0               0.617830894441751    0.4722230808231802   0.38171501412669406
     2026-07-29 00:26:40  15.065 sec  15.0               0.5109753197147372   0.3784870497227261   0.26109577735757794
     2026-07-29 00:26:41  15.251 sec  20.0               0.44111368507007787  0.3156783019546691   0.19458128315610382
     2026-07-29 00:26:41  15.461 sec  25.0               0.4067439923962636   0.284500376285093    0.1654406753504517
     2026-07-29 00:26:41  15.668 sec  30.0               0.382227147884688    0.2624152441257563   0.14609759258006313
     2026-07-29 00:26:41  15.860 sec  35.0               0.3651895115355381   0.24789120218978403  0.1333633793355649
     2026-07-29 00:26:41  16.046 sec  40.0               0.34783585389151034  0.23473247741922482  0.12098978125243612
     2026-07-29 00:26:42  16.241 sec  45.0               0.3358994029907933   0.22632615955146831  0.11282840892957134
---  ---                  ---         ---                ---                  ---                  ---
     2026-07-29 00:26:42  16.627 sec  55.0               0.3176689889023286   0.21408124998465275  0.10091358651022775
     2026-07-29 00:26:42  16.831 sec  60.0               

In [10]:
# Show leaderboard
df_leader_reg = aml_reg.leaderboard.as_data_frame()
print(df_leader_reg.head())

                        model_id      rmse       mse       mae     rmsle  \
0  GBM_4_AutoML_1_20260729_02346  0.455131  0.207145  0.296762  0.136820   
1  GBM_2_AutoML_1_20260729_02346  0.455992  0.207929  0.300417  0.137703   
2  GBM_3_AutoML_1_20260729_02346  0.458074  0.209832  0.301015  0.137618   
3  GBM_1_AutoML_1_20260729_02346  0.459199  0.210863  0.302926  0.138295   
4  GBM_5_AutoML_1_20260729_02346  0.460024  0.211622  0.305412  0.139356   

   mean_residual_deviance  
0                0.207145  
1                0.207929  
2                0.209832  
3                0.210863  
4                0.211622  


/usr/local/lib/python3.12/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


In [11]:
# Evaluate on test
perf_reg = aml_reg.leader.model_performance(test)
print(f"H2O Regression R²: {perf_reg.r2():.4f}")
print(f"H2O Regression RMSE: {perf_reg.rmse():.4f}")

H2O Regression R²: 0.8523
H2O Regression RMSE: 0.4417


## 4. Classification Example: Breast Cancer Dataset

In [12]:
from sklearn.datasets import load_breast_cancer

In [13]:
# Load and prepare dataset
data_cls = load_breast_cancer(as_frame=True)
Xc = data_cls.data
yc = data_cls.target.rename('target')

In [14]:
df_cls = h2o.H2OFrame(pd.concat([Xc, yc], axis=1))
train_cls, test_cls = df_cls.split_frame(ratios=[0.7], seed=42)

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


### 4.1 Run H2O AutoML for Classification

In [15]:
aml_cls = H2OAutoML(
    max_runtime_secs=300,
    max_models=20,
    seed=42,
    nfolds=5,
    balance_classes=True,
    project_name="breast_cancer_classification"
)
aml_cls.train(x=Xc.columns.tolist(), y='target', training_frame=train_cls)

AutoML progress: |
00:28:50.161: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training.

█
00:28:53.884: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training.


00:28:54.340: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training.

█
00:28:55.571: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training.

███
00:28:58.378: _response param, We have d

key,value
Stacking strategy,cross_validation
Number of base models (used / total),4/5
# GBM base models (used / total),1/1
# XGBoost base models (used / total),0/1
# DRF base models (used / total),2/2
# GLM base models (used / total),1/1
Metalearner algorithm,GLM
Metalearner fold assignment scheme,Random
Metalearner nfolds,5
Metalearner fold_column,None


In [16]:
# Show leaderboard
df_leader_cls = aml_cls.leaderboard.as_data_frame()
print(df_leader_cls.head())

                                            model_id      rmse       mse  \
0  StackedEnsemble_BestOfFamily_1_AutoML_2_202607...  0.182021  0.033132   
1  StackedEnsemble_AllModels_1_AutoML_2_20260729_...  0.184126  0.033903   
2        GBM_grid_1_AutoML_2_20260729_02850_model_22  0.185965  0.034583   
3        GBM_grid_1_AutoML_2_20260729_02850_model_11  0.186716  0.034863   
4        GBM_grid_1_AutoML_2_20260729_02850_model_20  0.188528  0.035543   

        mae     rmsle  mean_residual_deviance  
0  0.102690  0.130640                0.033132  
1  0.097524  0.132021                0.033903  
2  0.096905  0.131893                0.034583  
3  0.089201  0.134120                0.034863  
4  0.090711  0.133354                0.035543  


/usr/local/lib/python3.12/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


In [17]:
# Evaluate on test
perf_cls = aml_cls.leader.model_performance(test_cls)
print(f"H2O Classification: {perf_cls}")

H2O Classification: ModelMetricsRegressionGLM: stackedensemble
** Reported on test data. **

MSE: 0.037043105199522024
RMSE: 0.19246585463276863
MAE: 0.1050351571585801
RMSLE: 0.1322812239323879
Mean Residual Deviance: 0.037043105199522024
R^2: 0.8355508273586023
Null degrees of freedom: 177
Residual degrees of freedom: 173
Null deviance: 40.43221198186829
Residual deviance: 6.593672725514921
AIC: -69.48768312783018


MSE (Mean Squared Error):
- Measures the average squared difference between predicted and actual values. Lower values indicate better performance.


RMSE (Root Mean Squared Error):
- The square root of MSE, providing an error metric in the same units as the target variable. Lower values are better.


MAE (Mean Absolute Error):
-Measures the average absolute difference between predicted and actual values. Lower values indicate better performance.


RMSLE (Root Mean Squared Logarithmic Error):
- Similar to RMSE but uses the logarithm of the values, making it more robust to outliers. Lower values are better.


Mean Residual Deviance:
- Measures the goodness of fit of the model. Lower values indicate a better fit.


R² (Coefficient of Determination):
- Represents the proportion of variance explained by the model. Values closer to 1 indicate better performance.


Null Degrees of Freedom:
- The number of observations minus 1.


Residual Degrees of Freedom:
- The number of observations minus the number of parameters estimated.


Null Deviance:
- The deviance of the null model (model with no predictors).


Residual Deviance:
- The deviance of the fitted model. Lower values indicate a better fit.


AIC (Akaike Information Criterion):
- A measure of model quality, balancing goodness of fit and model complexity. Lower values indicate a better model.

## 5. Interpreting Results
- **Leaderboard** displays model ranking by default metric.
- Use `model_performance` to compute custom metrics (RMSE, R², AUC, accuracy, etc.).
- Top models are automatically ensembled by H2O's Stacked Ensemble.

## 6. AutoML Best Practices
- **Set time and model limits** (`max_runtime_secs`, `max_models`) to control cost and runtime.
- **Use cross-validation** (`nfolds`) for robust performance estimates.
- **Balance classes** for imbalanced classification tasks.
- **Review variable importance** on the leader model: `aml.leader.varimp()`.
- **Save and deploy** the best model: `h2o.save_model(aml.leader, path='best_model')`.

## 7. Exercise: Custom Dataset AutoML

**Task:** Apply H2O AutoML to a custom dataset.

1. Load any tabular dataset (CSV or from `sklearn.datasets`).
2. Decide whether it's a regression or classification task.
3. Initialize H2O and convert to `H2OFrame`.
4. Split into appropriate train/test ratios.
5. Run `H2OAutoML` with:
   - `max_runtime_secs=300`
   - `max_models=15`
   - `nfolds=5`
   - `balance_classes=True` (if classification)
6. Display the leaderboard and evaluate on the test set using relevant metrics.
7. Save the leaderboard to a pandas DataFrame and export it as `leaderboard.csv`

In [18]:
from sklearn.datasets import load_diabetes

# =====================================================
# Step 1: Load Dataset
# =====================================================

diabetes = load_diabetes(as_frame=True)
df = diabetes.frame

print("Dataset Shape:", df.shape)
display(df.head())

Dataset Shape: (442, 11)


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


# =====================================================
# Step 2: Determine the Task Type
# =====================================================
The Diabetes dataset is a Regression task.

Why?

The target column is target, which contains continuous numerical values, not categories.

In [19]:
# =====================================================
# Step 3: Initialize H2O and Convert to H2OFrame
# =====================================================

h2o.init()

hf = h2o.H2OFrame(df)

# Features and Target
x = [col for col in hf.columns if col != "target"]
y = "target"

Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,6 mins 30 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.11
H2O_cluster_version_age:,2 months and 6 days
H2O_cluster_name:,H2O_from_python_unknownUser_2avmm0
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,1.851 Gb
H2O_cluster_total_cores:,2
H2O_cluster_allowed_cores:,2
H2O_cluster_status:,"locked, healthy"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [20]:
# =====================================================
# Step 4: Split Dataset (80% Train, 20% Test)
# =====================================================

train, test = hf.split_frame(ratios=[0.8], seed=42)

print("Training Rows:", train.nrows)
print("Testing Rows:", test.nrows)

Training Rows: 343
Testing Rows: 99


In [21]:
# =====================================================
# Step 5: Train H2O AutoML
# =====================================================

aml = H2OAutoML(
    max_runtime_secs=300,
    max_models=15,
    nfolds=5,
    seed=42
)

aml.train(
    x=x,
    y=y,
    training_frame=train
)

AutoML progress: |███████████████████████████████████████████████████████████████| (done) 100%


key,value
Stacking strategy,cross_validation
Number of base models (used / total),4/15
# GBM base models (used / total),3/6
# XGBoost base models (used / total),0/5
# GLM base models (used / total),1/1
# DRF base models (used / total),0/2
# DeepLearning base models (used / total),0/1
Metalearner algorithm,GLM
Metalearner fold assignment scheme,Random
Metalearner nfolds,5


In [22]:
# =====================================================
# Step 6: Leaderboard & Model Evaluation
# =====================================================

print("Leaderboard:")
print(aml.leaderboard)

best_model = aml.leader

print("\nBest Model:")
print(best_model.model_id)

performance = best_model.model_performance(test)

print("\nRegression Metrics")
print("----------------------")
print("RMSE :", performance.rmse())
print("MSE  :", performance.mse())
print("MAE  :", performance.mae())
print("R²   :", performance.r2())

Leaderboard:
model_id                                                   rmse      mse      mae     rmsle    mean_residual_deviance
StackedEnsemble_AllModels_1_AutoML_3_20260729_03003     55.5494  3085.73  45.6097  0.424879                   3085.73
StackedEnsemble_BestOfFamily_1_AutoML_3_20260729_03003  55.6148  3093     45.8143  0.427846                   3093
GLM_1_AutoML_3_20260729_03003                           55.6304  3094.74  45.5883  0.43039                    3094.74
GBM_3_AutoML_3_20260729_03003                           56.5214  3194.67  46.124   0.432413                   3194.67
GBM_grid_1_AutoML_3_20260729_03003_model_1              56.7001  3214.9   46.064   0.428702                   3214.9
GBM_4_AutoML_3_20260729_03003                           57.372   3291.54  46.3214  0.437312                   3291.54
GBM_2_AutoML_3_20260729_03003                           57.6424  3322.64  46.6963  0.438363                   3322.64
GBM_1_AutoML_3_20260729_03003                  

In [23]:
# =====================================================
# Step 7: Save Leaderboard
# =====================================================

leaderboard_df = aml.leaderboard.as_data_frame()

leaderboard_df.to_csv("leaderboard.csv", index=False)

print("\nLeaderboard saved as leaderboard.csv")


Leaderboard saved as leaderboard.csv


/usr/local/lib/python3.12/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


In [24]:
# Display leaderboard
display(leaderboard_df)

,model_id,rmse,mse,mae,rmsle,mean_residual_deviance
0,StackedEnsemble_AllModels_1_AutoML_3_20260729_...,55.549366,3085.732067,45.609696,0.424879,3085.732067
1,StackedEnsemble_BestOfFamily_1_AutoML_3_202607...,55.614788,3093.004672,45.814267,0.427846,3093.004672
2,GLM_1_AutoML_3_20260729_03003,55.630372,3094.738250,45.588339,0.430390,3094.738250
3,GBM_3_AutoML_3_20260729_03003,56.521374,3194.665720,46.123980,0.432413,3194.665720
4,GBM_grid_1_AutoML_3_20260729_03003_model_1,56.700131,3214.904893,46.063954,0.428702,3214.904893
5,GBM_4_AutoML_3_20260729_03003,57.371959,3291.541635,46.321363,0.437312,3291.541635
6,GBM_2_AutoML_3_20260729_03003,57.642376,3322.643478,46.696259,0.438363,3322.643478
7,GBM_1_AutoML_3_20260729_03003,57.966407,3360.104378,47.656061,0.441148,3360.104378
8,DRF_1_AutoML_3_20260729_03003,58.212727,3388.721572,47.895172,0.441174,3388.721572
9,XRT_1_AutoML_3_20260729_03003,58.315365,3400.681790,47.600991,0.436633,3400.681790


In [25]:
# Download (Google Colab only)
from google.colab import files
files.download("leaderboard.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>